In [ ]:
import os
import requests
import zipfile
import shutil
import tarfile
from tqdm import tqdm

data_url = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip"
zip_path = "caltech-101.zip"
extract_path = "./caltech-101-data" 

def download_file(url, filename):
    print(f"Đang tải {filename}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(filename, "wb") as f, tqdm(
        total=total_size, unit='iB', unit_scale=True, desc=filename
    ) as pbar:
        for data in response.iter_content(1024):
            size = f.write(data)
            pbar.update(size)

if not os.path.exists(zip_path):
    download_file(data_url, zip_path)

print("Đang giải nén file ZIP chính...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

inner_files = {
    "101_ObjectCategories.tar.gz": os.path.join(extract_path, "caltech-101", "101_ObjectCategories.tar.gz"),
    "Annotations.tar": os.path.join(extract_path, "caltech-101", "Annotations.tar")
}

for name, path in inner_files.items():
    if os.path.exists(path):
        print(f"Đang giải nén {name}...")
        shutil.unpack_archive(path, ".") 
    else:
        print(f"Cảnh báo: Không tìm thấy {path}")

path_images = "./101_ObjectCategories/airplanes/"
path_annot = "./Annotations/Airplanes_Side_2/"

if os.path.exists(path_images) and os.path.exists(path_annot):
    print("\n✅ Dữ liệu đã sẵn sàng tại:")
    print(f"- Images: {path_images}")
    print(f"- Annotations: {path_annot}")

Đang giải nén file ZIP chính...
Đang giải nén 101_ObjectCategories.tar.gz...
Đang giải nén Annotations.tar...

✅ Dữ liệu đã sẵn sàng tại:
- Images: ./101_ObjectCategories/airplanes/
- Annotations: ./Annotations/Airplanes_Side_2/


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import numpy as np

# Cấu hình các tham số
patch_size = 32
image_size = 224
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 4
transformer_units = [projection_dim * 2, projection_dim]
transformer_layers = 4
mlp_head_units = [2048, 1024, 512, 64, 32] 

# 1. Trích xuất Patches
class Patches(nn.Module):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        # Sử dụng unfold để cắt ảnh thành các patch
        patches = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        # patches: [B, C, num_h, num_w, p_h, p_w]
        patches = patches.contiguous().view(B, C, -1, self.patch_size, self.patch_size)
        patches = patches.permute(0, 2, 3, 4, 1) # [B, num_patches, p_h, p_w, C]
        return patches.flatten(2) # [B, num_patches, p_h*p_w*C]

# 2. Patch Encoder (Linear Projection + Positioning)
class PatchEncoder(nn.Module):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = nn.Linear(patch_size * patch_size * 3, projection_dim)
        self.position_embedding = nn.Embedding(num_patches, projection_dim)

    def forward(self, patch):
        positions = torch.arange(0, self.num_patches, step=1).to(patch.device)
        projected_patches = self.projection(patch)
        encoded = projected_patches + self.position_embedding(positions)
        return encoded

# 3. Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, projection_dim, num_heads, dropout=0.1):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(projection_dim)
        self.mha = nn.MultiheadAttention(projection_dim, num_heads, dropout=dropout, batch_first=True)
        self.layernorm2 = nn.LayerNorm(projection_dim)
        self.mlp = nn.Sequential(
            nn.Linear(projection_dim, projection_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim * 2, projection_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x1 = self.layernorm1(x)
        attn_output, _ = self.mha(x1, x1, x1)
        x2 = x + attn_output
        x3 = self.layernorm2(x2)
        return x2 + self.mlp(x3)

# 4. Mô hình ViT Object Detector hoàn chỉnh
class ViTObjectDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_layer = Patches(patch_size)
        self.patch_encoder = PatchEncoder(num_patches, projection_dim)
        
        self.transformer_layers = nn.Sequential(
            *[TransformerBlock(projection_dim, num_heads) for _ in range(transformer_layers)]
        )
        
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(projection_dim),
            nn.Flatten(),
            nn.Linear(num_patches * projection_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 4), # Output: [x1, y1, x2, y2]
            nn.Sigmoid() # Giới hạn tọa độ trong khoảng [0, 1]
        )

    def forward(self, x):
        patches = self.patch_layer(x)
        encoded = self.patch_encoder(patches)
        features = self.transformer_layers(encoded)
        return self.mlp_head(features)

In [ ]:
import os
import scipy.io
import torch
import cv2
from torch.utils.data import Dataset
from torchvision import transforms

class CaltechViTDataset(Dataset):
    def __init__(self, img_dir, annot_dir, img_size=224):
        self.img_dir = img_dir
        self.annot_dir = annot_dir
        self.img_size = img_size
        
        # Lấy danh sách file và sort để đảm bảo ảnh khớp với annot
        self.img_names = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.annot_names = sorted([f for f in os.listdir(annot_dir) if f.endswith('.mat')])
        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # 1. Đọc ảnh
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape
        
        # 2. Đọc annotation (.mat)
        annot_path = os.path.join(self.annot_dir, self.annot_names[idx])
        annot = scipy.io.loadmat(annot_path)["box_coord"][0]
        
        # Tọa độ trong Caltech 101 thường là [y1, y2, x1, x2]
        # Ta chuẩn hóa về [0, 1] để model dễ học (chia cho w, h)
        top_left_x, top_left_y = annot[2] / w, annot[0] / h
        bottom_right_x, bottom_right_y = annot[3] / w, annot[1] / h
        
        target = torch.tensor([top_left_x, top_left_y, bottom_right_x, bottom_right_y], dtype=torch.float32)
        
        # 3. Transform ảnh
        if self.transform:
            image = self.transform(image)
            
        return image, target

In [ ]:
from dataset import CaltechViTDataset
from model import ViTObjectDetector # Model mình đã viết ở câu trả lời trước
from torch.utils.data import DataLoader

# Khởi tạo dataset
train_ds = CaltechViTDataset(
    img_dir="./101_ObjectCategories/airplanes/",
    annot_dir="./Annotations/Airplanes_Side_2/"
)

# DataLoader: Giúp load dữ liệu theo batch, xáo trộn (shuffle) và dùng đa nhân CPU
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# Khởi tạo model và optimizer
model = ViTObjectDetector().to("cuda")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

# Loop huấn luyện
for epoch in range(50):
    for images, targets in train_loader:
        images, targets = images.to("cuda"), targets.to("cuda")
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1} - Loss: {loss.item():.4f}")

In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import transforms
from PIL import Image

# Đảm bảo device đã được định nghĩa (quan trọng khi chạy trên Kaggle)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Chuyển mô hình sang chế độ evaluation và đẩy lên device
model.to(device)
model.eval()

# 2. Đường dẫn ảnh (Sửa lỗi tên biến ở đây)
img_path = "/kaggle/working/101_ObjectCategories/airplanes/image_0009.jpg"

# 3. Tiền xử lý
image_size = 224
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

# Mở ảnh và lấy kích thước gốc
original_image = Image.open(img_path).convert("RGB") # Đã sửa image_path thành img_path
width, height = original_image.size

# Transform và thêm batch dimension: [3, 224, 224] -> [1, 3, 224, 224]
input_tensor = transform(original_image).unsqueeze(0).to(device)

# 4. Dự đoán
with torch.no_grad():
    prediction = model(input_tensor)
    # Lấy kết quả, đưa về CPU và chuyển sang numpy
    box = prediction.cpu().squeeze().numpy()

# 5. Giải mã tọa độ (Denormalize)
# Model trả về tỉ lệ [0, 1], ta nhân với pixel thực tế
x1, y1, x2, y2 = box[0] * width, box[1] * height, box[2] * width, box[3] * height

# 6. Hiển thị
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(original_image)

# Vẽ khung hình chữ nhật
# Lưu ý: patches.Rectangle nhận (x, y), width, height
rect = patches.Rectangle(
    (x1, y1), x2 - x1, y2 - y1, 
    linewidth=3, edgecolor='r', facecolor='none'
)
ax.add_patch(rect)

plt.title(f"Dự đoán ViT: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]", color='red')
plt.axis('off')
plt.show()